## **Libraries**

In [ ]:
import numpy as np
import tensorflow as tf

c:\python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## **1. Write a program to implement logical gates AND, OR and NOT with McCulloch-Pitts**

**What is McCulloh-Pitts?**


In [ ]:
def mcp_neuron(inputs, weights, threshold):
    weighted_sum = sum(i * w for i, w in zip(inputs, weights))
    return 1 if weighted_sum >= threshold else 0

In [ ]:
def AND(x1, x2):
    return mcp_neuron([x1, x2], [1, 1], 2)

def OR(x1, x2):
    return mcp_neuron([x1, x2], [1, 1], 1)

def NOT(x):
    return mcp_neuron([x], [-1], 0)

In [ ]:
print("AND Gate")
for x1 in [0, 1]:
    for x2 in [0, 1]:
        print(x1, x2, "->", AND(x1, x2))

print("\nOR Gate")
for x1 in [0, 1]:
    for x2 in [0, 1]:
        print(x1, x2, "->", OR(x1, x2))

print("\nNOT Gate")
for x in [0, 1]:
    print(x, "->", NOT(x))

AND Gate
0 0 -> 0
0 1 -> 0
1 0 -> 0
1 1 -> 1

OR Gate
0 0 -> 0
0 1 -> 1
1 0 -> 1
1 1 -> 1

NOT Gate
0 -> 1
1 -> 0


## **2. Write a program to implement Hebb‟s rule.**

In [ ]:
import numpy as np

# Hebbian Learning Function
def hebbian_learning(inputs, targets, learning_rate=1.0):
    # Number of features
    n_features = inputs.shape[1]

    # Initialize weights to zero
    weights = np.zeros(n_features)

    # Training
    for i in range(len(inputs)):
        x = inputs[i]
        y = targets[i]

    return weights + learning_rate * x * y # Hebbian update

# Example dataset (AND logic)
inputs = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

targets = np.array([0, 0, 0, 1])

# Convert 0 → -1 (common in Hebbian learning)
inputs = np.where(inputs == 0, -1, 1)
targets = np.where(targets == 0, -1, 1)

# Train
weights = hebbian_learning(inputs, targets)

print("Final Weights:", weights)

Final Weights: [2. 2.]


## **3. To implement Kohonen Self-Organizing Maps Code**

In [1]:
import numpy as np

class SOM:
    def __init__(self, x, y, input_dim, learning_rate=0.5, sigma=None, iterations=1000):
        self.x = x
        self.y = y
        self.input_dim = input_dim
        self.learning_rate = learning_rate
        self.iterations = iterations

        if sigma is None:
            sigma = max(x, y) / 2.0
        self.sigma = sigma

        # Initialize weight vectors randomly
        self.weights = np.random.rand(x, y, input_dim)

        # Coordinate grid for neurons
        self._init_neuron_locations()

    def _init_neuron_locations(self):
        self.neuron_locations = np.array(
            [[i, j] for i in range(self.x) for j in range(self.y)]
        )

    def _euclidean_distance(self, x, w):
        return np.linalg.norm(x - w, axis=-1)

    def _find_bmu(self, sample):
        distances = np.linalg.norm(self.weights - sample, axis=2)
        bmu_index = np.unravel_index(np.argmin(distances), (self.x, self.y))
        return bmu_index

    def _decay_learning_rate(self, t):
        return self.learning_rate * np.exp(-t / self.iterations)

    def _decay_sigma(self, t):
        return self.sigma * np.exp(-t / self.iterations)

    def _neighborhood(self, bmu, sigma):
        d = np.sum((self.neuron_locations - np.array(bmu))**2, axis=1)
        return np.exp(-d / (2 * (sigma ** 2))).reshape(self.x, self.y)

    def train(self, data):
        for t in range(self.iterations):
            sample = data[np.random.randint(0, len(data))]

            bmu = self._find_bmu(sample)

            lr = self._decay_learning_rate(t)
            sigma = self._decay_sigma(t)

            neighborhood = self._neighborhood(bmu, sigma)

            # Update weights
            for i in range(self.x):
                for j in range(self.y):
                    influence = neighborhood[i, j]
                    self.weights[i, j] += lr * influence * (sample - self.weights[i, j])

    def map_vects(self, data):
        mapping = []
        for sample in data:
            mapping.append(self._find_bmu(sample))
        return mapping

In [2]:
# Sample dataset (2D)
data = np.array([
    [0.1, 0.2],
    [0.3, 0.2],
    [0.8, 0.7],
    [0.9, 0.6],
    [0.4, 0.5]
])

# Create SOM (5x5 grid, input dimension = 2)
som = SOM(x=5, y=5, input_dim=2, learning_rate=0.5, iterations=2000)

# Train SOM
som.train(data)

# Map data to BMUs
mapped = som.map_vects(data)

print("BMU mapping:")
for i, m in enumerate(mapped):
    print(f"Data {data[i]} -> BMU {m}")

BMU mapping:
Data [0.1 0.2] -> BMU (np.int64(4), np.int64(0))
Data [0.3 0.2] -> BMU (np.int64(0), np.int64(0))
Data [0.8 0.7] -> BMU (np.int64(2), np.int64(4))
Data [0.9 0.6] -> BMU (np.int64(0), np.int64(3))
Data [0.4 0.5] -> BMU (np.int64(3), np.int64(2))


In [3]:
import numpy as np

class FuzzySet:
    def __init__(self, elements):
        """
        elements: dict {element: membership_value}
        """
        self.elements = elements

    # Union: max(μA, μB)
    def union(self, other):
        result = {}
        all_keys = set(self.elements.keys()).union(other.elements.keys())

        for x in all_keys:
            mu_a = self.elements.get(x, 0)
            mu_b = other.elements.get(x, 0)
            result[x] = max(mu_a, mu_b)

        return FuzzySet(result)

    # Intersection: min(μA, μB)
    def intersection(self, other):
        result = {}
        all_keys = set(self.elements.keys()).union(other.elements.keys())

        for x in all_keys:
            mu_a = self.elements.get(x, 0)
            mu_b = other.elements.get(x, 0)
            result[x] = min(mu_a, mu_b)

        return FuzzySet(result)

    # Complement: 1 - μA
    def complement(self):
        result = {x: 1 - mu for x, mu in self.elements.items()}
        return FuzzySet(result)

    # Difference: A - B = A ∩ (1 - B)
    def difference(self, other):
        comp_b = other.complement()
        return self.intersection(comp_b)

    def display(self):
        print(self.elements)

In [4]:
# Define fuzzy sets
A = FuzzySet({
    "x1": 0.2,
    "x2": 0.7,
    "x3": 0.5
})

B = FuzzySet({
    "x1": 0.6,
    "x2": 0.4,
    "x3": 0.8
})

# Operations
print("A:")
A.display()

print("\nB:")
B.display()

print("\nUnion:")
A.union(B).display()

print("\nIntersection:")
A.intersection(B).display()

print("\nComplement of A:")
A.complement().display()

print("\nDifference (A - B):")
A.difference(B).display()

A:
{'x1': 0.2, 'x2': 0.7, 'x3': 0.5}

B:
{'x1': 0.6, 'x2': 0.4, 'x3': 0.8}

Union:
{'x3': 0.8, 'x1': 0.6, 'x2': 0.7}

Intersection:
{'x3': 0.5, 'x1': 0.2, 'x2': 0.4}

Complement of A:
{'x1': 0.8, 'x2': 0.30000000000000004, 'x3': 0.5}

Difference (A - B):
{'x3': 0.19999999999999996, 'x1': 0.2, 'x2': 0.6}
